In [2]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to the synthetic e-commerce database we generated earlier
conn = sqlite3.connect('/Users/manushdesai/Desktop/multi-agent/data/ecommerce.db')  # adjust path to match where your file actually is

# We JOIN sales_history with products so we get category and price alongside
# each day's sales — we'll need these as model inputs (features) later
sales = pd.read_sql("""
    SELECT sh.product_id, sh.sale_date, sh.units_sold, sh.is_promo,
           p.category, p.price
    FROM sales_history sh
    JOIN products p ON sh.product_id = p.product_id
""", conn)



# Dates come back as text from SQLite — convert to real datetime objects
# so we can do date math (day of week, sorting, etc.) later
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

# Sort by product first, then date — this order matters a lot for the next
# step, where we compute "yesterday's sales" per product. If it's not sorted
# correctly, our lag features will be wrong.
sales = sales.sort_values(['product_id', 'sale_date']).reset_index(drop=True)

sales.head()

,product_id,sale_date,units_sold,is_promo,category,price
0,1,2025-08-01,20,0,Electronics,862.66
1,1,2025-08-02,18,0,Electronics,862.66
2,1,2025-08-03,38,0,Electronics,862.66
3,1,2025-08-04,12,0,Electronics,862.66
4,1,2025-08-05,18,0,Electronics,862.66


In [3]:
grouped_sold = sales.groupby('product_id')['units_sold']
shifted = grouped_sold.shift(1)

sales['lag_1'] = shifted
sales['lag_7'] = grouped_sold.shift(7)

sales['roll_mean_7']  = shifted.groupby(sales['product_id']).transform(lambda s: s.rolling(7).mean())
sales['roll_mean_14'] = shifted.groupby(sales['product_id']).transform(lambda s: s.rolling(14).mean())
sales['roll_mean_30'] = shifted.groupby(sales['product_id']).transform(lambda s: s.rolling(30).mean())
sales['roll_std_7']   = shifted.groupby(sales['product_id']).transform(lambda s: s.rolling(7).std())

sales['trend_14'] = sales['roll_mean_7'] - sales['roll_mean_14']

sales[['product_id','sale_date','units_sold','lag_1','lag_7','roll_mean_7','roll_mean_30']].tail(10)

,product_id,sale_date,units_sold,lag_1,lag_7,roll_mean_7,roll_mean_30
29190,80,2026-07-22,20,11.0,8.0,16.285714,17.633333
29191,80,2026-07-23,17,20.0,14.0,18.000000,17.400000
29192,80,2026-07-24,16,17.0,16.0,18.428571,17.266667
29193,80,2026-07-25,14,16.0,23.0,18.428571,17.066667
29194,80,2026-07-26,18,14.0,21.0,17.142857,16.600000
29195,80,2026-07-27,15,18.0,21.0,16.714286,16.533333
29196,80,2026-07-28,11,15.0,11.0,15.857143,16.133333
29197,80,2026-07-29,15,11.0,20.0,15.857143,16.033333
29198,80,2026-07-30,16,15.0,17.0,15.142857,16.066667
29199,80,2026-07-31,7,16.0,16.0,15.000000,16.366667


In [4]:
# --- Calendar features ---
# These help the model learn patterns tied to time itself — like the
# weekend sales bump we verified earlier in this project.

sales['day_of_week'] = sales['sale_date'].dt.dayofweek
sales['is_weekend'] = (sales['day_of_week'] >= 5).astype(int)
sales['month'] = sales['sale_date'].dt.month
sales['day_of_month'] = sales['sale_date'].dt.day
# --- Encoding category ---
# Models only understand numbers, not text like "Electronics" or "Beauty".
# pd.get_dummies() converts each category into its own 0/1 column.
# drop_first=True drops one category to avoid redundancy (if a row is 0
# in every other category column, it must belong to the dropped one —
# keeping all columns would be duplicate information).
sales = pd.get_dummies(sales, columns=['category'], drop_first=True)

sales.columns.tolist()

['product_id',
 'sale_date',
 'units_sold',
 'is_promo',
 'price',
 'lag_1',
 'lag_7',
 'roll_mean_7',
 'roll_mean_14',
 'roll_mean_30',
 'roll_std_7',
 'trend_14',
 'day_of_week',
 'is_weekend',
 'month',
 'day_of_month',
 'category_Beauty',
 'category_Electronics',
 'category_Home & Kitchen',
 'category_Sports']

In [5]:
# --- Cleaning up ---
# The first ~30 days of each product's history don't have enough "past"
# to compute lag_7, roll_mean_30, etc. — those cells are NaN (missing).
# A model can't train on NaN values, so we drop these rows.
# This is expected and normal — we're not losing real data, just rows
# where we don't have enough history yet to make a fair prediction.

print("Rows before dropping:", len(sales))

sales = sales.dropna(subset=['lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_std_7'])

print("Rows after dropping:", len(sales))

Rows before dropping: 29200
Rows after dropping: 26800


In [6]:
sales.head()

,product_id,sale_date,units_sold,is_promo,price,lag_1,lag_7,roll_mean_7,roll_mean_14,roll_mean_30,roll_std_7,trend_14,day_of_week,is_weekend,month,day_of_month,category_Beauty,category_Electronics,category_Home & Kitchen,category_Sports
30,1,2025-08-31,15,0,862.66,24.0,20.0,15.714286,18.785714,20.566667,6.421689,-3.071429,6,1,8,31,False,True,False,False
31,1,2025-09-01,14,0,862.66,15.0,5.0,15.000000,18.357143,20.400000,6.137318,-3.357143,0,0,9,1,False,True,False,False
32,1,2025-09-02,11,0,862.66,14.0,13.0,16.285714,18.142857,20.266667,4.386125,-1.857143,1,0,9,2,False,True,False,False
33,1,2025-09-03,72,0,862.66,11.0,18.0,16.000000,17.357143,19.366667,4.690416,-1.357143,2,0,9,3,False,True,False,False
34,1,2025-09-04,94,0,862.66,72.0,11.0,23.714286,20.714286,21.366667,21.784660,3.000000,3,0,9,4,False,True,False,False


In [7]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26800 entries, 30 to 29199
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   product_id               26800 non-null  int64         
 1   sale_date                26800 non-null  datetime64[ns]
 2   units_sold               26800 non-null  int64         
 3   is_promo                 26800 non-null  int64         
 4   price                    26800 non-null  float64       
 5   lag_1                    26800 non-null  float64       
 6   lag_7                    26800 non-null  float64       
 7   roll_mean_7              26800 non-null  float64       
 8   roll_mean_14             26800 non-null  float64       
 9   roll_mean_30             26800 non-null  float64       
 10  roll_std_7               26800 non-null  float64       
 11  trend_14                 26800 non-null  float64       
 12  day_of_week              26800 non-n

In [8]:
sales.tail()

,product_id,sale_date,units_sold,is_promo,price,lag_1,lag_7,roll_mean_7,roll_mean_14,roll_mean_30,roll_std_7,trend_14,day_of_week,is_weekend,month,day_of_month,category_Beauty,category_Electronics,category_Home & Kitchen,category_Sports
29195,80,2026-07-27,15,0,1378.88,18.0,21.0,16.714286,17.214286,16.533333,3.450328,-0.500000,0,0,7,27,False,True,False,False
29196,80,2026-07-28,11,0,1378.88,15.0,11.0,15.857143,16.642857,16.133333,2.911390,-0.785714,1,0,7,28,False,True,False,False
29197,80,2026-07-29,15,0,1378.88,11.0,20.0,15.857143,16.071429,16.033333,2.911390,-0.214286,2,0,7,29,False,True,False,False
29198,80,2026-07-30,16,0,1378.88,15.0,17.0,15.142857,16.571429,16.066667,2.267787,-1.428571,3,0,7,30,False,True,False,False
29199,80,2026-07-31,7,0,1378.88,16.0,16.0,15.000000,16.714286,16.366667,2.160247,-1.714286,4,0,7,31,False,True,False,False


In [9]:
cutoff_date = sales['sale_date'].quantile(0.8, interpolation='nearest')

train = sales[sales['sale_date'] < cutoff_date]
test = sales[sales['sale_date'] >= cutoff_date]

print("Cutoff date:", cutoff_date)
print("Train rows:", len(train), " | Train date range:", train['sale_date'].min(), "to", train['sale_date'].max())
print("Test rows:", len(test), " | Test date range:", test['sale_date'].min(), "to", test['sale_date'].max())

Cutoff date: 2026-05-25 00:00:00
Train rows: 21360  | Train date range: 2025-08-31 00:00:00 to 2026-05-24 00:00:00
Test rows: 5440  | Test date range: 2026-05-25 00:00:00 to 2026-07-31 00:00:00


In [10]:
feature_cols = (
    ['lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_std_7', 'trend_14',
     'day_of_week', 'is_weekend', 'month', 'day_of_month', 'price', 'is_promo']
    + [c for c in sales.columns if c.startswith('category_')]
)

X_train, y_train = train[feature_cols], train['units_sold']
X_test, y_test = test[feature_cols], test['units_sold']

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 17
['lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_std_7', 'trend_14', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'price', 'is_promo', 'category_Beauty', 'category_Electronics', 'category_Home & Kitchen', 'category_Sports']


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf_v2 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf_v2.fit(X_train, y_train)
rf_pred = rf_v2.predict(X_test)

print("Both models trained.")
print("Sample Linear Regression predictions:", lr_pred[:5])
print("Sample Random Forest predictions:", rf_pred[:5])
print("Actual values:", y_test[:5].values)

Both models trained.
Sample Linear Regression predictions: [26.22195447 23.90612739 26.01546743 27.97687298 27.97774687]
Sample Random Forest predictions: [28.60612806 28.57586026 28.57771892 28.5472261  28.57376861]
Actual values: [15 23 40 35 39]


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:30s}  MAE: {mae:6.2f}   RMSE: {rmse:6.2f}")
    return mae, rmse

naive_pred = test['lag_1']

print("--- Performance on TEST set (unseen future dates) ---")
evaluate("Naive baseline (predict lag_1)", y_test, naive_pred)
evaluate("Linear Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)

rf_train_pred = rf_v2.predict(X_train)
print("\n--- Overfitting check (Random Forest) ---")
evaluate("Random Forest on TRAINING data", y_train, rf_train_pred)
evaluate("Random Forest on TEST data", y_test, rf_pred)

--- Performance on TEST set (unseen future dates) ---
Naive baseline (predict lag_1)  MAE:  10.13   RMSE:  17.09
Linear Regression               MAE:   8.38   RMSE:  13.41
Random Forest                   MAE:   8.32   RMSE:  13.84

--- Overfitting check (Random Forest) ---
Random Forest on TRAINING data  MAE:   5.00   RMSE:   7.84
Random Forest on TEST data      MAE:   8.32   RMSE:  13.84


(8.31625310928627, np.float64(13.841018036983188))

In [13]:
rf_v2 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf_v2.fit(X_train, y_train)

rf_v2_train_pred = rf_v2.predict(X_train)
rf_v2_test_pred = rf_v2.predict(X_test)

print("--- max_depth=6 (more restricted) ---")
evaluate("RF v2 on TRAINING data", y_train, rf_v2_train_pred)
evaluate("RF v2 on TEST data", y_test, rf_v2_test_pred)

--- max_depth=6 (more restricted) ---
RF v2 on TRAINING data          MAE:   5.00   RMSE:   7.84
RF v2 on TEST data              MAE:   8.32   RMSE:  13.84


(8.31625310928627, np.float64(13.841018036983186))

In [14]:
importances = pd.Series(rf_v2.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("--- Feature importance (Random Forest, max_depth=6) ---")
print(importances.head(10))

--- Feature importance (Random Forest, max_depth=6) ---
roll_mean_30            0.829976
is_promo                0.045845
day_of_week             0.036023
is_weekend              0.033669
lag_1                   0.010528
roll_mean_14            0.007788
category_Electronics    0.006398
roll_std_7              0.005690
day_of_month            0.004826
trend_14                0.003575
dtype: float64


In [15]:
import joblib
import os

save_dir = '/Users/manushdesai/Desktop/multi-agent/models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(rf_v2, f'{save_dir}/demand_model.pkl')

with open(f'{save_dir}/demand_model_features.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print("Model saved to:", save_dir)

Model saved to: /Users/manushdesai/Desktop/multi-agent/models


In [16]:
print(os.path.exists('/Users/manushdesai/Desktop/multi-agent/models/demand_model.pkl'))
print(os.listdir('/Users/manushdesai/Desktop/multi-agent/models'))


True
['demand_model.pkl', 'returns_model_features.txt', 'returns_model.pkl', 'demand_model_features.txt']
